In [1]:
import pandas as pd
import re
import numpy as np

In [2]:
df=pd.read_csv("IMDB Dataset.csv")

In [3]:
df.head()
df.isnull().sum()

review       0
sentiment    0
dtype: int64

In [4]:
df.shape

(50000, 2)

In [5]:
df.drop_duplicates(inplace=True)

In [6]:
df.shape

(49582, 2)

Data Preprocessing

#1.CONVERTING TO LOWER CASE

In [7]:
df["review"]=df["review"].str.lower()

#2.REMOVING URLS

In [8]:
def removeURL(text):
    text=re.sub(r"https\S+","",text)
    return text
df["review"]=df["review"].apply(removeURL)

#3.REMOVING PUNCTUATION

In [9]:
def removePunctuation(text):
    text=re.sub(r"^[A-Za-z0-9]","",text)
    return text
df["review"]=df["review"].apply(removePunctuation)

#4.REMOVING HTML TAGS

In [10]:
def removeHTML(text):
    text=re.sub(r"<.*?","",text)
    return text
df["review"]=df["review"].apply(removeHTML)

#5.REMOVING STOPWARDS

In [11]:
import nltk
# nltk.download("punkt")
# nltk.download("punkt_tab")
# nltk.download("stopwords")
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
def removeStopwords(text):
    tokens=word_tokenize(text)
    stop_words=stopwords.words("english")
    for word in tokens:
        if(word in stop_words):
            text=text.replace(word,"")
    return text
df["review"]=df["review"].apply(removeStopwords)

#6.STEMMING(coding->code,coder->code,etc..)

In [12]:
from nltk.stem import PorterStemmer
def stemming(text):
    ps=PorterStemmer()
    stemmed_words=[]
    tokens=word_tokenize(text)
    for token in tokens:
        stemmed_token=ps.stem(token)
        stemmed_words.append(stemmed_token)
    texts=" ".join(stemmed_words)
    return texts
df["review"]=df["review"].apply(stemming)

#7.ENCODING OF OUTPUT VALUE

In [13]:
from sklearn.preprocessing import LabelEncoder
le=LabelEncoder()
df["sentiment"]=le.fit_transform(df["sentiment"])
y=df["sentiment"]

#8.VECTURIZATION USING TF-IDF

In [14]:
from sklearn.feature_extraction.text import TfidfVectorizer
vectorizer=TfidfVectorizer(max_features=5000)
X=vectorizer.fit_transform(df["review"])

In [15]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2,stratify=y, random_state=42
)

In [16]:
#converting X_train and X_text into numpy array
X_train=X_train.toarray()
X_test=X_test.toarray()

In [17]:
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset,DataLoader
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train.values, dtype=torch.float32).reshape(-1,1)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test.values, dtype=torch.float32).reshape(-1,1)

train_set = TensorDataset(X_train_tensor, y_train_tensor)
test_set = TensorDataset(X_test_tensor, y_test_tensor)
train_loader = DataLoader(train_set, batch_size=64, shuffle=True)
test_loader = DataLoader(test_set, batch_size=64)

Building RNN model

In [18]:
import torch.optim as optim
class RNN(nn.Module):
    def __init__(self,input_size,hidden_size=128,num_layer=1): #hidden_size=no of hidden layers and num_layer=no of RNN layers 
        super().__init__()
        self.hidden_size=hidden_size
        self.num_layer=num_layer
        
        #RNN Layer
        self.rnn=nn.RNN(input_size,hidden_size,num_layer,batch_first=True)
        #FullyConnected layer
        #as this is many to one architecture here hidden_size=no.of neurons or no of hidden layer and output=1
        self.fc_layer=nn.Linear(hidden_size,1)
        
    def forward(self,x):
        #Hidden state=h0 shape(no of RNN layers,batch size,hidden size)
        #torch.zer0s=> initialize h0 =0
        h0=torch.zeros(self.num_layer,x.size(0),self.hidden_size)
        out,_=self.rnn(x,h0)# it returns two output
        #1st output=hidden state of all timestates=>(batch,seuence_len,hidden_size)
        #2nd output=final hidden state of last timestate
        out=self.fc_layer(out[:,-1,:])
        #out[:,-1,:]=> returns last timestate
        return out

In [19]:
model=RNN(X_train.shape[1]) #X_train.shape[1]=input size
criterion=nn.BCELoss()
optimizer=optim.Adam(model.parameters())

TRAIN RNN USING UNSQUEEZE AND SQUZEE FUNCTION FOR RESHAPING INPUT
UNSQUEEZE=> add 1 additional dimension 
SQUEEZE=> reduce 1 dimension

In [20]:
epochs = 10
for epoch in range(epochs):
    model.train()
    for xb, yb in train_loader:
        optimizer.zero_grad()
        xb = xb.unsqueeze(1)   # add singleton dimension
        # here use unsqueeze because tfidf return 2d vector but RNN expect 3D vector
        outputs = model(xb)
        outputs = torch.sigmoid(outputs.squeeze()) #outputs.squeeze()=>batch_size
        #used sigmoid function to convert output into probability which ius needed in sentiment analysis
        loss = criterion(outputs, yb.squeeze())
        loss.backward()
        optimizer.step()

    print(f"Epoch {epoch+1}, Loss: {loss.item():.4f}")

Epoch 1, Loss: 0.3495
Epoch 2, Loss: 0.3184
Epoch 3, Loss: 0.2416
Epoch 4, Loss: 0.3662
Epoch 5, Loss: 0.1133
Epoch 6, Loss: 0.2202
Epoch 7, Loss: 0.2163
Epoch 8, Loss: 0.2381
Epoch 9, Loss: 0.1797
Epoch 10, Loss: 0.1556


In [21]:
all_preds = []
all_labels = []

model.eval()

with torch.no_grad():

    for xb, yb in test_loader:

        xb = xb.unsqueeze(1)

        outputs = model(xb)

        preds = torch.sigmoid(outputs)

        preds = (preds > 0.5).float()

        all_preds.extend(preds.numpy())

        all_labels.extend(yb.numpy())


all_preds = np.array(all_preds)
all_labels = np.array(all_labels)

In [29]:
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
print(f"Accuracy: {accuracy_score(all_labels, all_preds)*100:.2f}%")
print("Confusion Matrix:\n", confusion_matrix(all_labels, all_preds))
print(classification_report(all_labels, all_preds))

Accuracy: 85.67%
Confusion Matrix:
 [[4215  725]
 [ 696 4281]]
              precision    recall  f1-score   support

         0.0       0.86      0.85      0.86      4940
         1.0       0.86      0.86      0.86      4977

    accuracy                           0.86      9917
   macro avg       0.86      0.86      0.86      9917
weighted avg       0.86      0.86      0.86      9917



In [28]:
model.eval()

correct = 0
total = 0

with torch.no_grad():
    for xb, yb in test_loader:
        xb = xb.unsqueeze(1)
        outputs = model(xb)
        preds = (torch.sigmoid(outputs) > 0.5).float()
        correct += (preds == yb).sum().item()
        total += yb.size(0)
accuracy = correct / total

print(f"Test Accuracy: {accuracy*100:.2f}%")

Test Accuracy: 85.67%
